# 02 - Model Training

Train and compare churn prediction models using the processed dataset from notebook 01.

## 1. Load Processed Data

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('D:\project\data\processed\clean_data.csv')

In [ ]:
df.shape

In [ ]:
X = df.drop('churn', axis=1)

In [ ]:
y = df['churn']

In [ ]:
print(X.shape)
print(y.shape)

## 2. Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y, test_size=0.2,random_state=42,stratify=y
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

## 3. Logistic Regression Baseline

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train,y_train)

In [ ]:
print("Model training completed.")

In [ ]:
y_pred = lr_model.predict(X_test)

print(y_pred[:10])

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
lr_model = LogisticRegression(random_state=42, class_weight='balanced')
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)
print(classification_report(y_test, y_pred))

## 4. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)
print(classification_report(y_test, y_pred_rf))

In [ ]:
rf_model = RandomForestClassifier(
    random_state=42, 
    class_weight='balanced',
    n_estimators=100
)

In [ ]:
rf_model.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)
print(classification_report(y_test, y_pred_rf))

## 5. XGBoost

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb_model = XGBClassifier(
    random_state=42,
    scale_pos_weight=9
)
xgb_model.fit(X_train, y_train)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

## 6. LightGBM and SMOTE Setup

Install optional packages if needed, train LightGBM, then prepare a balanced training set with SMOTE.

In [ ]:
!pip install lightgbm

In [ ]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    random_state=42,
    class_weight='balanced'
)
lgbm_model.fit(X_train, y_train)

y_pred_lgbm = lgbm_model.predict(X_test)
print(classification_report(y_test, y_pred_lgbm))

In [ ]:
!pip install imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(y_train_sm.value_counts())

## 7. Models Trained on SMOTE Data

In [ ]:
lr_model2 = LogisticRegression(random_state=42)
lr_model2.fit(X_train_sm, y_train_sm)

In [ ]:
y_pred_lr2 = lr_model2.predict(X_test)
print(classification_report(y_test, y_pred_lr2))

In [ ]:
xgb_model2 = XGBClassifier(random_state=42)
xgb_model2.fit(X_train_sm, y_train_sm)

In [ ]:
y_pred_xgb2 = xgb_model2.predict(X_test)
print(classification_report(y_test, y_pred_xgb2))

In [ ]:
lgbm_model2 = LGBMClassifier(random_state=42)
lgbm_model2.fit(X_train_sm, y_train_sm)

y_pred_lgbm2 = lgbm_model2.predict(X_test)
print(classification_report(y_test, y_pred_lgbm2))

## 8. Save Final Model and Compare Metrics

In [ ]:
import joblib

joblib.dump(lgbm_model, "D:\\project\\models\\lgbm_model.pkl")
print("Model saved!")

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, recall_score, precision_score

models = {
    'Logistic Regression': y_pred,
    'Random Forest': y_pred_rf,
    'XGBoost': y_pred_xgb,
    'LightGBM': y_pred_lgbm
}

for name, pred in models.items():
    print(f"\n{name}")
    print(f"  F1-Score : {f1_score(y_test, pred):.3f}")
    print(f"  Recall   : {recall_score(y_test, pred):.3f}")
    print(f"  Precision: {precision_score(y_test, pred):.3f}")
    print(f"  ROC-AUC  : {roc_auc_score(y_test, pred):.3f}")